In [ ]:
!pip install roboflow ultralytics

In [ ]:
!pip install tensorflow==2.21.0 keras==3.14.1

In [ ]:
from google.colab import drive
from google.colab import userdata

import json
import numpy as np
import tensorflow
from tensorflow.keras.layers import Flatten, Dense, ReLU, Dense, Activation
from pathlib import Path

In [ ]:
drive.mount('/content/drive')

project_directory_path = "/content/drive/MyDrive/Colab Notebooks/Projects/PAAI Project"
additional_dataset_directory_path = project_directory_path + "/additional_dataset"

dataset_cropped_directory_path = project_directory_path + "/dataset_cropped"
cropped_json = dataset_cropped_directory_path + "/project-2-at-2026-06-07-19-37-c29d30b5.json"

YOLO_MODEL_PATH = "/models/best_yolo.pt"
KERAS_MODEL_PATH = "/models/best_point.keras"

---

In [ ]:
RETRAIN = True  # set False to skip training and load saved model
IMG_SIZE   = (224, 224)
BATCH_SIZE = 16
VAL_SPLIT  = 0.2

In [ ]:
def parse_ls_export(json_path, images_dir):
    with open(json_path) as f:
        tasks = json.load(f)

    image_paths, labels = [], []

    for task in tasks:
        filename = Path(task["image"]).name
        path = str(Path(images_dir) / filename)

        points = {}
        for kp in task["keypoints"]:
            label = kp["keypointlabels"][0]
            points[label] = (kp["x"] / 100.0, kp["y"] / 100.0)

        try:
            row = [
                points["tl"][0], points["tl"][1],
                points["tr"][0], points["tr"][1],
                points["br"][0], points["br"][1],
                points["bl"][0], points["bl"][1],
            ]
        except KeyError:
            continue

        image_paths.append(path)
        labels.append(row)

    return image_paths, np.array(labels, dtype=np.float32)

In [ ]:
image_paths, labels = parse_ls_export(cropped_json, dataset_cropped_directory_path)
print(f"{len(image_paths)} labeled images")

In [ ]:
def load_and_preprocess(path, label):
    img = tensorflow.io.read_file(path)
    img = tensorflow.image.decode_jpeg(img, channels=3)
    img = tensorflow.image.resize(img, IMG_SIZE)
    img = tensorflow.keras.applications.resnet50.preprocess_input(img)
    return img, label

n = len(image_paths)
indices = np.random.permutation(n)
val_idx, train_idx = indices[:int(n*VAL_SPLIT)], indices[int(n*VAL_SPLIT):]

def make_ds(idx, shuffle=False):
    ds = tensorflow.data.Dataset.from_tensor_slices(
        ([image_paths[i] for i in idx], labels[idx])
    )
    ds = ds.map(load_and_preprocess, num_parallel_calls=tensorflow.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(512)
    return ds.batch(BATCH_SIZE).prefetch(tensorflow.data.AUTOTUNE)

In [ ]:
train_ds = make_ds(train_idx, shuffle=True)
val_ds = make_ds(val_idx)

In [ ]:
imagenet_base = tensorflow.keras.applications.ResNet50(weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3))
imagenet_base.trainable = False

In [ ]:
model = tensorflow.keras.Sequential([
    imagenet_base,
    Flatten(),
    Dense(256),
    ReLU(),
    Dense(8),
    # Activation("sigmoid"),
])

In [ ]:
model.compile(optimizer=tensorflow.keras.optimizers.Adam(1e-4), loss=tensorflow.keras.losses.Huber())
model.summary()

In [ ]:
callbacks = [
    tensorflow.keras.callbacks.ModelCheckpoint("best.keras", monitor="val_loss", save_best_only=True),
    tensorflow.keras.callbacks.EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True),
    tensorflow.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4),
]

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

In [ ]:
model.get_layer("resnet50").trainable = True
model.compile(optimizer=tensorflow.keras.optimizers.Adam(1e-5), loss=tensorflow.keras.losses.Huber())

In [ ]:
model.fit(train_ds, validation_data=val_ds, epochs=50, callbacks=callbacks)

In [ ]:
if RETRAIN:
    model.save(KERAS_MODEL_PATH)

In [ ]:
if not RETRAIN:
    model = tensorflow.keras.models.load_model(KERAS_MODEL_PATH)

In [ ]:
import cv2
from PIL import Image
import random

img_path = random.choice(image_paths)

img = tensorflow.keras.utils.load_img(img_path, target_size=IMG_SIZE)
img = tensorflow.keras.utils.img_to_array(img)
img = tensorflow.keras.applications.resnet50.preprocess_input(img)
img = tensorflow.expand_dims(img, axis=0)

pred = model.predict(img)
pred = np.clip(pred, 0, 1)